# ESM-DB Download Workflow

**Author:** Spina Cianetti  
**License:** [GPL-3.0](https://www.gnu.org/licenses/gpl-3.0.html)  
This code is released under the GNU General Public License v3.0.

---

Downloads all events from the ESM database in ASDF format together with metadata CSV files.

**Adaptive strategy per event:**
- Attempts a **single-call** download first (all data types in one request)
- If the server returns **413** (too many records) → automatically switches to
  **per-initial-letter batch** mode (`&station=A*`, `&station=B*,C*`, ...),
  downloads temporary files and merges them into a single ASDF
- Files already present on disk are **skipped** automatically (safe resume)

**Output per event:**
- `{event_id}.h5` — ASDF containing waveforms, StationXML, AuxiliaryData (spectra)
- `{event_id}_SA.csv` — flatfile metadata + SA response spectra
- `{event_id}_SD.csv` — flatfile metadata + SD response spectra

**Sections:**
1. Imports and configuration
2. Event list download (year by year)
3. Helper functions
4. Main download loop
5. Completeness check

## 1. Imports and configuration

In [1]:
import os
import csv
import glob
import time
import shutil
import requests
import io

import pandas as pd
import pyasdf
from obspy import Catalog
from tqdm.notebook import tqdm

# ── Paths ─────────────────────────────────────────────────────────────────────
OUTPUT_DIR = '/home/jovyan/shared/users/spina/ESM25/output'
TMP_BASE   = os.path.join(OUTPUT_DIR, '_tmp_downloads')  # temporary batch files
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TMP_BASE,   exist_ok=True)

EVENTS_FILE   = 'event_list.txt'    # generated in section 2
ERROR_LOG_CSV = 'error_log.csv'

# ── Web service endpoints ─────────────────────────────────────────────────────
BASE_URL     = 'https://esm-db.eu/esmws/eventdata/1/query'
FLATFILE_URL = 'https://esm-db.eu/esmws/flatfile/1/query'
FDSNWS_URL   = 'https://esm-db.eu/fdsnws/event/1/query'

MAX_RECORDS       = 1000   # ESM limit per single request
MAX_STA_PER_BATCH = MAX_RECORDS // 3  # 333 stations × 3 components = 999 records
SLEEP_S           = 5      # pause between successive downloads (seconds)

# ── Year range ────────────────────────────────────────────────────────────────
YEAR_START = 1967
YEAR_END   = 2026

# ── Data types to download ────────────────────────────────────────────────────
DOWNLOAD_CONFIG = [
    ('CV', 'ACC', 'acc_cv'),
    ('MP', 'ACC', 'acc_mp'),
    ('MP', 'VEL', 'vel_mp'),
    ('MP', 'DIS', 'dis_mp'),
    ('MP', 'SA',  'sa_mp'),
    ('MP', 'SD',  'sd_mp'),
]

print(f'OUTPUT_DIR : {OUTPUT_DIR}')
print(f'TMP_BASE   : {TMP_BASE}')

OUTPUT_DIR : /home/jovyan/shared/users/spina/ESM25/output
TMP_BASE   : /home/jovyan/shared/users/spina/ESM25/output/_tmp_downloads


## 2. Event list download

Queries the ESM FDSNWS service to retrieve all event IDs, one year at a time.  
Skipped if `event_list.txt` already exists.

**Failure handling:**
- On timeout or HTTP error, a single automatic retry is attempted with a longer timeout (90 s)
- Years that fail even after the retry are collected and saved to `failed_years.txt`
- Re-run only the failed years by replacing `YEAR_START`/`YEAR_END` with the values in that file
  (delete `event_list.txt` first so the cell does not skip)

In [2]:
FAILED_YEARS_FILE = 'failed_years.txt'  # years whose download failed and need a re-run


def fetch_year(year, timeout=30):
    """Download the event list for a single year from the FDSNWS service.
    Returns the raw text content on success, or raises an exception on failure.
    """
    url = (
        f'{FDSNWS_URL}'
        f'?starttime={year}-01-01T00:00:00'
        f'&endtime={year}-12-31T23:59:59'
        f'&format=text'
    )
    resp = requests.get(url, timeout=timeout)
    resp.raise_for_status()
    return resp.text.strip()


if os.path.exists(EVENTS_FILE):
    with open(EVENTS_FILE) as f:
        n = sum(1 for l in f if l.strip())
    print(f'[SKIP] {EVENTS_FILE} already present ({n} events)')
else:
    raw_file   = 'all_events.txt'
    failed_years = []   # years that failed even after retry

    print('Downloading event list year by year...')

    with open(raw_file, 'w') as out_f:
        for year in tqdm(range(YEAR_START, YEAR_END + 1), desc='Years', unit='year'):
            success = False
            last_err = None

            # First attempt (timeout=30 s)
            try:
                content = fetch_year(year, timeout=30)
                if content:
                    out_f.write(f'# ==== {year} ====\n{content}\n\n')
                success = True
            except Exception as e:
                last_err = e
                tqdm.write(f'  ⚠️  {year}: {e} — retrying with timeout=90 s …')
                time.sleep(5)

            # Single retry with a longer timeout
            if not success:
                try:
                    content = fetch_year(year, timeout=90)
                    if content:
                        out_f.write(f'# ==== {year} ====\n{content}\n\n')
                    success = True
                    tqdm.write(f'  ✅  {year}: retry succeeded')
                except Exception as e:
                    last_err = e
                    tqdm.write(f'  ❌  {year}: retry also failed — {e}')
                    failed_years.append(year)

            time.sleep(1)

    # Save the list of failed years so they can be re-run independently
    if failed_years:
        with open(FAILED_YEARS_FILE, 'w') as f:
            for y in failed_years:
                f.write(str(y) + '\n')
        print(f'\n⚠️  {len(failed_years)} year(s) could not be downloaded: {failed_years}')
        print(f'   → list saved to {FAILED_YEARS_FILE} for a targeted re-run')
    else:
        # Clean up the failed-years file if a previous run had left one
        if os.path.exists(FAILED_YEARS_FILE):
            os.remove(FAILED_YEARS_FILE)

    # Extract event IDs (first column, '|' separator)
    with open(raw_file) as f_in, open(EVENTS_FILE, 'w') as f_out:
        for line in f_in:
            if not line.strip() or line.startswith('#') or line.startswith('ev_id'):
                continue
            ev_id = line.split('|')[0].strip()
            if ev_id:
                f_out.write(ev_id + '\n')

    with open(EVENTS_FILE) as f:
        n = sum(1 for l in f if l.strip())
    print(f'✅ {EVENTS_FILE} saved ({n} events)')

Years:   0%|          | 0/60 [00:00<?, ?year/s]

  ⚠️  1969: HTTPSConnectionPool(host='esm-db.eu', port=443): Read timed out. (read timeout=30) — retrying with timeout=90 s …
  ✅  1969: retry succeeded
  ⚠️  1975: HTTPSConnectionPool(host='esm-db.eu', port=443): Read timed out. (read timeout=30) — retrying with timeout=90 s …
  ❌  1975: retry also failed — HTTPSConnectionPool(host='esm-db.eu', port=443): Read timed out. (read timeout=90)
  ⚠️  1978: HTTPSConnectionPool(host='esm-db.eu', port=443): Read timed out. (read timeout=30) — retrying with timeout=90 s …
  ✅  1978: retry succeeded
  ⚠️  1987: HTTPSConnectionPool(host='esm-db.eu', port=443): Read timed out. (read timeout=30) — retrying with timeout=90 s …
  ✅  1987: retry succeeded
  ⚠️  1993: HTTPSConnectionPool(host='esm-db.eu', port=443): Read timed out. (read timeout=30) — retrying with timeout=90 s …
  ✅  1993: retry succeeded
  ⚠️  2001: HTTPSConnectionPool(host='esm-db.eu', port=443): Read timed out. (read timeout=30) — retrying with timeout=90 s …
  ✅  2001: retry succe

## 3. Helper functions

In [3]:
# ── Streaming download with progress bar ─────────────────────────────────────

def stream_download(url, output_path, label):
    """Download a file via HTTP streaming with a tqdm progress bar.
    Returns (ok: bool, http_status: int).
    """
    try:
        with requests.get(url, stream=True, timeout=300) as resp:
            if resp.status_code != 200:
                return False, resp.status_code
            total = int(resp.headers.get('Content-Length', 0))
            with open(output_path, 'wb') as f, tqdm(
                desc=label,
                total=total if total > 0 else None,
                unit='B', unit_scale=True, unit_divisor=1024,
                leave=False,
            ) as bar:
                for chunk in resp.iter_content(chunk_size=1024 * 1024):
                    f.write(chunk)
                    bar.update(len(chunk))
        size = os.path.getsize(output_path)
        if size > 1024:
            return True, 200
        os.remove(output_path)   # file too small = likely a text error response
        return False, 200
    except Exception:
        if os.path.exists(output_path):
            os.remove(output_path)
        return False, -1


def download_with_retry(url, output_path, label, max_retries=3):
    """Retry wrapper around stream_download.
    Does not retry on HTTP 413 (pointless to retry without changing the request).
    """
    for attempt in range(1, max_retries + 1):
        ok, status = stream_download(url, output_path, label)
        if ok:
            return True, status
        if status == 413:
            return False, 413
        if attempt < max_retries:
            time.sleep(5)
    return False, status


# ── Per-initial-letter station batching ───────────────────────────────────────

def get_station_letter_batches(event_id):
    """Query the flatfile service to build station wildcard batches.
    Groups station initial letters so that each batch stays within
    MAX_STA_PER_BATCH (333 stations × 3 components < 1000 records).
    Returns a list of wildcard strings like ['A*', 'B*,C*,D*', ...]
    or None if the flatfile cannot be reached.
    """
    try:
        resp = requests.get(
            FLATFILE_URL,
            params={'eventid': event_id, 'bad-quality': 'Y',
                    'include-fields': 'network_code,station_code'},
            timeout=60,
        )
        if resp.status_code != 200 or not resp.text.strip():
            return None
        df = pd.read_csv(io.StringIO(resp.text), sep=';')
        df['first_letter'] = df['station_code'].astype(str).str[0].str.upper()
        by_letter = df.groupby('first_letter')['station_code'].nunique().sort_index()
    except Exception:
        return None

    batches = []
    current_group, current_count = [], 0
    for letter, count in by_letter.items():
        if count > MAX_STA_PER_BATCH:
            # Letter with too many stations: dedicated batch
            if current_group:
                batches.append(','.join(f'{l}*' for l in current_group))
                current_group, current_count = [], 0
            batches.append(f'{letter}*')
        elif current_count + count > MAX_STA_PER_BATCH:
            # Current group full: close and open a new one
            batches.append(','.join(f'{l}*' for l in current_group))
            current_group, current_count = [letter], count
        else:
            current_group.append(letter)
            current_count += count
    if current_group:
        batches.append(','.join(f'{l}*' for l in current_group))
    return batches


# ── ASDF merge ────────────────────────────────────────────────────────────────

def merge_asdf_files(output_path, input_files):
    """Merge a list of ASDF files into a single output file.
    Handles waveforms, StationXML, events and 3-level AuxiliaryData
    (e.g. Spectra/NET_STA/trace_name).
    """
    with pyasdf.ASDFDataSet(output_path, mode='w', mpi=False) as merged_ds:
        event_added = False
        for file_path in input_files:
            if not os.path.exists(file_path):
                continue
            with pyasdf.ASDFDataSet(file_path, mode='r', mpi=False) as ds:

                # Waveforms
                for station in ds.waveforms.list():
                    for tag in ds.waveforms[station].get_waveform_tags():
                        try:
                            merged_ds.add_waveforms(ds.waveforms[station][tag], tag=tag)
                        except Exception:
                            pass  # silent duplicate

                # StationXML
                for station in ds.waveforms.list():
                    try:
                        merged_ds.add_stationxml(ds.waveforms[station].StationXML)
                    except Exception:
                        pass

                # Events (added only once)
                if not event_added and ds.events:
                    try:
                        merged_ds.add_quakeml(ds.events)
                        event_added = True
                    except Exception as e:
                        tqdm.write(f'  ⚠️  Events: {e}')

                # AuxiliaryData — 3-level structure (type / path / key)
                try:
                    if hasattr(ds, 'auxiliary_data') and ds.auxiliary_data is not None:
                        for aux_type in ds.auxiliary_data.list():
                            for path in ds.auxiliary_data[aux_type].list():
                                for key in ds.auxiliary_data[aux_type][path].list():
                                    aux_obj = ds.auxiliary_data[aux_type][path][key]
                                    already = (
                                        aux_type in merged_ds.auxiliary_data and
                                        path     in merged_ds.auxiliary_data[aux_type] and
                                        key      in merged_ds.auxiliary_data[aux_type][path]
                                    )
                                    if not already:
                                        try:
                                            merged_ds.add_auxiliary_data(
                                                data=aux_obj.data[()],
                                                data_type=aux_type,
                                                path=f'{path}/{key}',
                                                parameters=dict(aux_obj.parameters),
                                            )
                                        except Exception:
                                            pass
                except Exception:
                    pass


# ── Single-call HDF5 download ─────────────────────────────────────────────────

def download_h5_simple(event_id, output_path):
    """Download the ASDF file for an event in a single request (all data types)."""
    url = (
        f'{BASE_URL}?eventid={event_id}'
        f'&processing-type=CV,MP'
        f'&data-type=ACC,VEL,DIS,SA,SD'
        f'&add-xml=True&add-auxiliary-data=True'
    )
    return download_with_retry(url, output_path, f'{event_id}.h5')


# ── Batched HDF5 download + merge ─────────────────────────────────────────────

def download_h5_batched(event_id, output_path, csv_writer):
    """Download the ASDF file in per-letter-initial batches, then merge.
    Called automatically when the single-call attempt returns HTTP 413.
    """
    letter_batches = get_station_letter_batches(event_id)
    if not letter_batches:
        tqdm.write(f'  ⚠️  {event_id}: no stations found in flatfile')
        return False

    tmp_dir = os.path.join(TMP_BASE, event_id)
    os.makedirs(tmp_dir, exist_ok=True)

    downloaded_tmp = []
    any_failed = False
    total_jobs = len(DOWNLOAD_CONFIG) * len(letter_batches)

    with tqdm(total=total_jobs, desc=f'  batch {event_id}', unit='file', leave=False) as pbar:
        for proc_type, data_type, tag in DOWNLOAD_CONFIG:
            for b_idx, sta_wc in enumerate(letter_batches):
                suffix   = f'_b{b_idx}' if len(letter_batches) > 1 else ''
                tmp_path = os.path.join(tmp_dir, f'{event_id}_{tag}{suffix}.h5')

                # Skip if already downloaded (resume support)
                if os.path.exists(tmp_path) and os.path.getsize(tmp_path) > 1024:
                    downloaded_tmp.append(tmp_path)
                    pbar.update(1)
                    continue

                url = (
                    f'{BASE_URL}?eventid={event_id}'
                    f'&processing-type={proc_type}'
                    f'&data-type={data_type}'
                    f'&station={sta_wc}'
                    f'&add-xml=True&add-auxiliary-data=True'
                )
                ok, status = download_with_retry(url, tmp_path, os.path.basename(tmp_path))
                if ok:
                    downloaded_tmp.append(tmp_path)
                else:
                    any_failed = True
                    csv_writer.writerow([event_id, os.path.basename(tmp_path), f'HTTP {status}'])

                pbar.update(1)
                time.sleep(SLEEP_S)

    if not downloaded_tmp:
        return False

    merge_asdf_files(output_path, downloaded_tmp)

    # Remove temporary files only if all downloads succeeded and the merged file is valid
    if not any_failed and os.path.exists(output_path) and os.path.getsize(output_path) > 1024:
        shutil.rmtree(tmp_dir, ignore_errors=True)

    return os.path.exists(output_path) and os.path.getsize(output_path) > 1024


# ── Flatfile (SA / SD) download ───────────────────────────────────────────────

def download_flatfile(event_id, output_path, spectra_type, csv_writer):
    """Download the metadata flatfile for an event (SA or SD spectra)."""
    params = f'eventid={event_id}&bad-quality=Y'
    if spectra_type == 'SD':
        params += '&spectra=SD'
    url = f'{FLATFILE_URL}?{params}'
    ok, status = download_with_retry(url, output_path, os.path.basename(output_path))
    if not ok:
        csv_writer.writerow([event_id, f'CSV_{spectra_type}', f'HTTP {status}'])
    return ok


print('✅ All functions defined')

✅ All functions defined


## 4. Main download loop

For each event in the list:
1. Skip if all three output files already exist (**automatic resume**)
2. Attempt ASDF download with a **single call**
3. If server returns **413** → switch to **per-letter batch** download + merge
4. Download **SA** and **SD** flatfiles independently

In [ ]:
with open(EVENTS_FILE) as f:
    event_ids = [l.strip() for l in f if l.strip()]
print(f'Events in list: {len(event_ids)}')

n_skip    = 0
n_ok      = 0
n_batched = 0
n_failed  = 0

with open(ERROR_LOG_CSV, 'w', newline='') as log_f:
    csv_writer = csv.writer(log_f)
    csv_writer.writerow(['event_id', 'file_type', 'error'])

    for event_id in tqdm(event_ids, desc='Downloading events', unit='event'):
        h5_path = os.path.join(OUTPUT_DIR, f'{event_id}.h5')
        sa_path = os.path.join(OUTPUT_DIR, f'{event_id}_SA.csv')
        sd_path = os.path.join(OUTPUT_DIR, f'{event_id}_SD.csv')

        h5_ok = os.path.exists(h5_path) and os.path.getsize(h5_path) > 1024
        sa_ok = os.path.exists(sa_path) and os.path.getsize(sa_path) > 1024
        sd_ok = os.path.exists(sd_path) and os.path.getsize(sd_path) > 1024

        # All files present: skip
        if h5_ok and sa_ok and sd_ok:
            n_skip += 1
            continue

        # ── ASDF (HDF5) ──────────────────────────────────────────────────────
        if not h5_ok:
            ok, status = download_h5_simple(event_id, h5_path)
            if ok:
                n_ok += 1
            elif status == 413:
                # Too many records: switch to per-letter batch mode
                tqdm.write(f'  ⚡ {event_id}: 413 → switching to batch mode')
                ok = download_h5_batched(event_id, h5_path, csv_writer)
                if ok:
                    n_batched += 1
                else:
                    n_failed += 1
                    csv_writer.writerow([event_id, 'H5_batch', 'merge failed or no files downloaded'])
            else:
                n_failed += 1
                csv_writer.writerow([event_id, 'H5', f'HTTP {status}'])

        # ── SA flatfile ──────────────────────────────────────────────────────
        if not sa_ok:
            download_flatfile(event_id, sa_path, 'SA', csv_writer)

        # ── SD flatfile ──────────────────────────────────────────────────────
        if not sd_ok:
            download_flatfile(event_id, sd_path, 'SD', csv_writer)

        time.sleep(SLEEP_S)

print('\n── Summary ─────────────────────────────────────')
print(f'Skipped (already present)  : {n_skip}')
print(f'Downloaded (single call)   : {n_ok}')
print(f'Downloaded (batch mode)    : {n_batched}')
print(f'Failed                     : {n_failed}')
print(f'Error log                  → {ERROR_LOG_CSV}')

Events in list: 8689


PT-1969-0001.h5:   0%|          | 0.00/391k [00:00<?, ?B/s]

PT-1969-0001_SA.csv:   0%|          | 0.00/6.76k [00:00<?, ?B/s]

PT-1969-0001_SD.csv:   0%|          | 0.00/6.87k [00:00<?, ?B/s]

IT-1972-0009.h5:   0%|          | 0.00/806k [00:00<?, ?B/s]

...

ME-1973-0001.h5:   0%|          | 0.00/77.9k [00:00<?, ?B/s]

ME-1973-0001_SA.csv:   0%|          | 0.00/4.69k [00:00<?, ?B/s]

ME-1973-0001_SD.csv:   0%|          | 0.00/4.69k [00:00<?, ?B/s]

BA-1974-0002.h5:   0%|          | 0.00/87.1k [00:00<?, ?B/s]

BA-1974-0002_SA.csv:   0%|          | 0.00/4.69k [00:00<?, ?B/s]

## 5. Completeness check

Identifies events with missing or empty files and saves the list for a targeted re-run.

In [9]:
with open(EVENTS_FILE) as f:
    event_ids = [l.strip() for l in f if l.strip()]

missing_h5, missing_sa, missing_sd = [], [], []

def _missing(path):
    return not os.path.exists(path) or os.path.getsize(path) <= 1024

for event_id in event_ids:
    if _missing(os.path.join(OUTPUT_DIR, f'{event_id}.h5')):
        missing_h5.append(event_id)
    if _missing(os.path.join(OUTPUT_DIR, f'{event_id}_SA.csv')):
        missing_sa.append(event_id)
    if _missing(os.path.join(OUTPUT_DIR, f'{event_id}_SD.csv')):
        missing_sd.append(event_id)

print(f'Total events in list   : {len(event_ids)}')
print(f'H5  missing/empty      : {len(missing_h5)}')
print(f'SA  missing/empty      : {len(missing_sa)}')
print(f'SD  missing/empty      : {len(missing_sd)}')

all_missing = sorted(set(missing_h5) | set(missing_sa) | set(missing_sd))
if all_missing:
    print(f'\nEvents with at least one missing file: {len(all_missing)}')
    print(f'  {"event_id":<30} {"H5":^5} {"SA":^5} {"SD":^5}')
    print(f'  {"-"*30} {"-"*5} {"-"*5} {"-"*5}')
    for ev in all_missing:
        h = '❌' if ev in missing_h5 else '✅'
        s = '❌' if ev in missing_sa else '✅'
        d = '❌' if ev in missing_sd else '✅'
        print(f'  {ev:<30} {h:^5} {s:^5} {d:^5}')
    with open('eventi_mancanti.txt', 'w') as f:
        for ev in all_missing:
            f.write(ev + '\n')
    print('\nList saved to eventi_mancanti.txt')
else:
    print('\n✅ All files are present and valid')

Total events in list   : 1
H5  missing/empty      : 0
SA  missing/empty      : 0
SD  missing/empty      : 0

✅ All files are present and valid
